In [1]:
import warnings
warnings.filterwarnings("ignore")
import random
import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sklearn.cluster import KMeans
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import RobustScaler

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D2 in response_OUS
data = list(OUS_D2['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D2, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D2 
clinical_train.isnull().sum().sum()

0

## Test dataset: MAASTRO 

In [6]:
(MAASTRO_D2['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [7]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [8]:
# need to choose patient_id from MAASTRO_D2 in response_MAASTRO
data = list(MAASTRO_D2['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 

In [9]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D2, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]

In [10]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [11]:
# Check if some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,shape_MinorAxisLength,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event


In [12]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

In [13]:
# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['OS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]
lower, upper = np.percentile(y_MAASTRO['OS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

X_train:  (139, 374)
y_train:  (139,)


(99, 376)

# Feature Selection: RENT

In [14]:
selected_features = ["shape_Sphericity",
                     "glrlm_HighGrayLevelRunEmphasis_PET_c04",
                     "shape_MajorAxisLength",
                     "LBP_102_PET"]

# Selecting features in the DataFrame
X_rent = X[selected_features]

In [15]:
# Selecting features in the DataFrame
X_rent = X[selected_features]
X_new = X_rent.copy()

X_MAASTRO_rent = X_MAASTRO[selected_features]
MAASTRO_new = X_MAASTRO_rent.copy()

# Standardization

In [16]:
# Standardize X_new, the new data with the selected features only 
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = RobustScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [17]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [18]:
X_new

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,0.761164,16.969770,42.073251,0.000000
1,0.697049,15.598394,24.613845,0.000000
2,0.565792,17.334294,48.030294,0.000034
3,0.684364,14.009277,25.589900,0.000000
4,0.503142,21.202180,34.684750,0.000199
...,...,...,...,...
134,0.742102,16.110383,33.069705,0.000000
135,0.722918,21.249575,41.043692,0.000000
136,0.652963,14.873884,36.618802,0.000000
137,0.724255,21.648860,45.870392,0.000000


In [19]:
X_new_std

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,0.704475,0.085656,0.059912,0.000000
1,0.101791,-0.291967,-0.755402,0.000000
2,-1.132016,0.186031,0.338093,0.524263
3,-0.017443,-0.729547,-0.709822,0.000000
4,-1.720923,1.251094,-0.285114,3.028668
...,...,...,...,...
134,0.525285,-0.150985,-0.360532,0.000000
135,0.344965,1.264145,0.011834,0.000000
136,-0.312609,-0.491468,-0.194798,0.000000
137,0.357529,1.374092,0.237230,0.000000


In [20]:
MAASTRO_new

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,0.668072,19.034156,50.002093,0.000026
1,0.669961,11.392444,41.753334,0.000167
2,0.624081,14.567421,44.375483,0.000057
3,0.577624,13.477331,46.115989,0.000000
4,0.630933,16.365554,54.394967,0.000000
...,...,...,...,...
94,0.671754,17.492295,34.218615,0.000000
95,0.632189,14.267900,51.046869,0.000000
96,0.645548,12.143752,50.417953,0.000000
97,0.727488,15.300229,44.901412,0.000000


In [21]:
MAASTRO_new_std

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,-0.170593,0.654106,0.430171,0.398737
1,-0.152832,-1.450119,0.044973,2.544568
2,-0.584101,-0.575856,0.167421,0.869691
3,-1.020797,-0.876024,0.248699,0.000000
4,-0.519690,-0.080721,0.635308,0.000000
...,...,...,...,...
94,-0.135979,0.229539,-0.306881,0.000000
95,-0.507883,-0.658333,0.478960,0.000000
96,-0.382318,-1.243239,0.449591,0.000000
97,0.387916,-0.374070,0.191981,0.000000


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [22]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 00:47:13,360] A new study created in memory with name: no-name-749ca9b7-deea-4d20-855f-60afdc744e99
python(81314) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6796536796536796
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7892156862745098


[I 2024-04-16 00:47:23,632] A new study created in memory with name: no-name-9687990c-ce7b-43ec-9436-3ba8e274b516


Fold 4 C-index: 0.7341772151898734
Fold 5 C-index: 0.6948356807511737
[I 2024-04-16 00:47:23,585] Trial 0 finished with value: 0.7358264523738474 and parameters: {}. Best is trial 0 with value: 0.7358264523738474.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7358264523738474], datetime_start=datetime.datetime(2024, 4, 16, 0, 47, 14, 81716), datetime_complete=datetime.datetime(2024, 4, 16, 0, 47, 23, 582168), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7358264523738474


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.17932951816141288
Fold 2 IBS: 0.15173912855888322
Fold 3 IBS: 0.17662964771229905
Fold 4 IBS: 0.1530370960050396
Fold 5 IBS: 0.19681553106978886
[I 2024-04-16 00:47:24,515] Trial 0 finished with value: 0.17151018430148474 and parameters: {}. Best is trial 0 with value: 0.17151018430148474.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.17151018430148474], datetime_start=datetime.datetime(2024, 4, 16, 0, 47, 23, 728500), datetime_complete=datetime.datetime(2024, 4, 16, 0, 47, 24, 515050), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.17151018430148474


In [23]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [24]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.736
train_ibs:  0.172


#### Test

In [25]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [26]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.567
IBS score: 0.259


In [27]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [28]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge 

#### Train

In [29]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 00:47:25,435] A new study created in memory with name: no-name-2c843a0a-8e37-4eda-9f4c-272b8ff0e351


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-16 00:47:25,670] A new study created in memory with name: no-name-ed6120db-60d8-4394-bd98-3eff16b20cda


Fold 1 C-index: 0.6471861471861472
Fold 2 C-index: 0.7700892857142857
Fold 3 C-index: 0.6299019607843137
Fold 4 C-index: 0.7109704641350211
Fold 5 C-index: 0.7183098591549296
[I 2024-04-16 00:47:25,655] Trial 0 finished with value: 0.6952915433949395 and parameters: {}. Best is trial 0 with value: 0.6952915433949395.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6952915433949395], datetime_start=datetime.datetime(2024, 4, 16, 0, 47, 25, 511133), datetime_complete=datetime.datetime(2024, 4, 16, 0, 47, 25, 655500), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6952915433949395


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.21397651654775943
Fold 2 IBS: 0.2215779056051601
Fold 3 IBS: 0.204535948534301
Fold 4 IBS: 0.2247380347558079
Fold 5 IBS: 0.2181243088160326
[I 2024-04-16 00:47:26,040] Trial 0 finished with value: 0.21659054285181217 and parameters: {}. Best is trial 0 with value: 0.21659054285181217.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.21659054285181217], datetime_start=datetime.datetime(2024, 4, 16, 0, 47, 25, 780920), datetime_complete=datetime.datetime(2024, 4, 16, 0, 47, 26, 39591), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.21659054285181217


In [30]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [31]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.695
train_ibs:  0.217


#### Test

In [32]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [33]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.577


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [34]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [35]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 00:47:26,323] A new study created in memory with name: no-name-03da26bf-2ea9-4f3f-aec9-575658b3632f


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7843137254901961


[I 2024-04-16 00:47:26,901] A new study created in memory with name: no-name-0f94f91e-e9c7-4158-b911-041d76bc9170


Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-16 00:47:26,883] Trial 0 finished with value: 0.7357337800920462 and parameters: {}. Best is trial 0 with value: 0.7357337800920462.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7357337800920462], datetime_start=datetime.datetime(2024, 4, 16, 0, 47, 26, 375365), datetime_complete=datetime.datetime(2024, 4, 16, 0, 47, 26, 882984), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7357337800920462


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.17984882789532766
Fold 2 IBS: 0.15097677823967964
Fold 3 IBS: 0.1765953973444983
Fold 4 IBS: 0.1535870472815473
Fold 5 IBS: 0.1957792713839434
[I 2024-04-16 00:47:27,322] Trial 0 finished with value: 0.1713574644289993 and parameters: {}. Best is trial 0 with value: 0.1713574644289993.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.1713574644289993], datetime_start=datetime.datetime(2024, 4, 16, 0, 47, 26, 939335), datetime_complete=datetime.datetime(2024, 4, 16, 0, 47, 27, 321129), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.1713574644289993


In [36]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [37]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.736
train_ibs:  0.171


#### Test 

In [38]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [39]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.566


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.257


In [40]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [41]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 00:47:27,845] A new study created in memory with name: no-name-ec6afeab-4a25-422b-9c32-8e15562e0ffb


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.7767857142857143
Fold 3 C-index: 0.7843137254901961
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-16 00:47:28,419] Trial 0 finished with value: 0.734840922949189 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.734840922949189.
Fold 1 C-index: 0.683982683982684
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-16 00:47:28,933] Trial 1 finished with value: 0.7358483713831081 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.7358483713831081.
Fold 1 C-index: 0.683982683982684
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-16 00:47:29,473] Trial 2 finished with value: 0.7358483713831081 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 1 with value: 

Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-16 00:47:43,095] Trial 24 finished with value: 0.7358483713831081 and parameters: {'l1_ratio': 0.21623254971460304}. Best is trial 10 with value: 0.7367141722489089.
Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-16 00:47:43,613] Trial 25 finished with value: 0.7367141722489089 and parameters: {'l1_ratio': 0.0853699617771228}. Best is trial 10 with value: 0.7367141722489089.
Fold 1 C-index: 0.683982683982684
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-16 00:47:44,081] Trial 26 finished with value: 0.7358483713831081 and parameters: {'l1_ratio': 0.17284350892750144}. Best is trial 10 with value: 0.7367141722489089.
Fold 1 C-index: 0.683982683982684
Fold 2 

Fold 5 C-index: 0.6948356807511737
[I 2024-04-16 00:47:57,842] Trial 48 finished with value: 0.7358483713831081 and parameters: {'l1_ratio': 0.28913086771769636}. Best is trial 10 with value: 0.7367141722489089.
Fold 1 C-index: 0.6796536796536796
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-16 00:47:58,351] Trial 49 finished with value: 0.7349825705173073 and parameters: {'l1_ratio': 0.3881372941956815}. Best is trial 10 with value: 0.7367141722489089.
Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.7130801687763713
Fold 5 C-index: 0.6948356807511737
[I 2024-04-16 00:47:59,018] Trial 50 finished with value: 0.7333386448227486 and parameters: {'l1_ratio': 0.02973857144887264}. Best is trial 10 with value: 0.7367141722489089.
Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index

...

Fold 1 IBS: 0.17958401971222676
Fold 2 IBS: 0.15098312924614873
Fold 3 IBS: 0.1767448712872295
Fold 4 IBS: 0.15363652262238406
Fold 5 IBS: 0.19586936134167535
[I 2024-04-16 00:48:31,294] Trial 0 finished with value: 0.17136358084193287 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.17136358084193287.
Fold 1 IBS: 0.1790606196014541
Fold 2 IBS: 0.1510332528697648
Fold 3 IBS: 0.17701163427407687
Fold 4 IBS: 0.15373342073867732
Fold 5 IBS: 0.19592625650922701
[I 2024-04-16 00:48:31,749] Trial 1 finished with value: 0.17135303679864003 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.17135303679864003.
Fold 1 IBS: 0.17896261637309524
Fold 2 IBS: 0.1510401822766344
Fold 3 IBS: 0.17706845641401528
Fold 4 IBS: 0.1537851323347694
Fold 5 IBS: 0.19593137200081384
[I 2024-04-16 00:48:32,362] Trial 2 finished with value: 0.17135755187986565 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 1 with value: 0.17135303679864

Fold 3 IBS: 0.17680302727562638
Fold 4 IBS: 0.15366552174881976
Fold 5 IBS: 0.19587730169037634
[I 2024-04-16 00:48:45,884] Trial 25 finished with value: 0.1713622910065737 and parameters: {'l1_ratio': 0.6017140536891199}. Best is trial 16 with value: 0.17134943011820747.
Fold 1 IBS: 0.17911828567036817
Fold 2 IBS: 0.15104768249515776
Fold 3 IBS: 0.1769985350004711
Fold 4 IBS: 0.15374697047778124
Fold 5 IBS: 0.1959384481298246
[I 2024-04-16 00:48:46,542] Trial 26 finished with value: 0.17136998435472056 and parameters: {'l1_ratio': 0.331298221798899}. Best is trial 16 with value: 0.17134943011820747.
Fold 1 IBS: 0.21310138293856212
Fold 2 IBS: 0.21969880624095167
Fold 3 IBS: 0.177272964741059
Fold 4 IBS: 0.22365673065934433
Fold 5 IBS: 0.19601497940733117
[I 2024-04-16 00:48:46,999] Trial 27 finished with value: 0.20594897279744967 and parameters: {'l1_ratio': 0.016560941611969082}. Best is trial 16 with value: 0.17134943011820747.
Fold 1 IBS: 0.17887368272381246
Fold 2 IBS: 0.15110187

Fold 1 IBS: 0.17877727213414701
Fold 2 IBS: 0.15109320806775142
Fold 3 IBS: 0.17716574753037861
Fold 4 IBS: 0.15385539803348391
Fold 5 IBS: 0.19598487862985922
[I 2024-04-16 00:49:02,309] Trial 50 finished with value: 0.17137530087912406 and parameters: {'l1_ratio': 0.11338510694936829}. Best is trial 16 with value: 0.17134943011820747.
Fold 1 IBS: 0.17900900942089837
Fold 2 IBS: 0.15103675947210204
Fold 3 IBS: 0.1770414645060716
Fold 4 IBS: 0.15376075899211938
Fold 5 IBS: 0.1959287812499061
[I 2024-04-16 00:49:03,077] Trial 51 finished with value: 0.1713553547282195 and parameters: {'l1_ratio': 0.2544539865445068}. Best is trial 16 with value: 0.17134943011820747.
Fold 1 IBS: 0.1790228218571628
Fold 2 IBS: 0.1510470888170803
Fold 3 IBS: 0.1770289703630868
Fold 4 IBS: 0.15374350822619956
Fold 5 IBS: 0.19594307104053688
[I 2024-04-16 00:49:03,547] Trial 52 finished with value: 0.17135709206081323 and parameters: {'l1_ratio': 0.26005323176164424}. Best is trial 16 with value: 0.171349430

Fold 5 IBS: 0.1959153936247194
[I 2024-04-16 00:49:15,934] Trial 74 finished with value: 0.17135049059792581 and parameters: {'l1_ratio': 0.3188475466506644}. Best is trial 16 with value: 0.17134943011820747.
Fold 1 IBS: 0.1791177750325778
Fold 2 IBS: 0.15102945844501012
Fold 3 IBS: 0.17697894913260018
Fold 4 IBS: 0.15371094557817447
Fold 5 IBS: 0.1959235266963892
[I 2024-04-16 00:49:16,328] Trial 75 finished with value: 0.17135213097695035 and parameters: {'l1_ratio': 0.3229533747185797}. Best is trial 16 with value: 0.17134943011820747.
Fold 1 IBS: 0.17917381462126708
Fold 2 IBS: 0.15102001885648014
Fold 3 IBS: 0.1769495304715724
Fold 4 IBS: 0.15374755942208862
Fold 5 IBS: 0.19591322044019194
[I 2024-04-16 00:49:16,818] Trial 76 finished with value: 0.17136082876232006 and parameters: {'l1_ratio': 0.3614250298741148}. Best is trial 16 with value: 0.17134943011820747.
Fold 1 IBS: 0.17930603170952508
Fold 2 IBS: 0.1509993843802153
Fold 3 IBS: 0.17688087927573307
Fold 4 IBS: 0.153716575

Fold 2 IBS: 0.15109000931956318
Fold 3 IBS: 0.17713983113321774
Fold 4 IBS: 0.15386028475603727
Fold 5 IBS: 0.19598488937913855
[I 2024-04-16 00:49:27,751] Trial 99 finished with value: 0.17138636515809855 and parameters: {'l1_ratio': 0.17481454265540544}. Best is trial 86 with value: 0.17134942947118037.


* Best trial for IBS: 
 FrozenTrial(number=86, state=TrialState.COMPLETE, values=[0.17134942947118037], datetime_start=datetime.datetime(2024, 4, 16, 0, 49, 21, 212099), datetime_complete=datetime.datetime(2024, 4, 16, 0, 49, 21, 773188), params={'l1_ratio': 0.31617649869393666}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'l1_ratio': FloatDistribution(high=1.0, log=False, low=0.0001, step=None)}, trial_id=86, value=None)


* Best Score for IBS: 
 0.17134942947118037


In [42]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [43]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.737
train_ibs:  0.171


#### Test

In [44]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [45]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.09799455604980911)

test_cindex : 0.566


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.31617649869393666)

test_ibs:  0.257


In [46]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [47]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 00:49:28,289] A new study created in memory with name: no-name-de583c67-596e-4471-bf45-c99c399fac31


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7467532467532467
Fold 2 C-index: 0.8125
Fold 3 C-index: 0.7769607843137255
Fold 4 C-index: 0.6814345991561181
Fold 5 C-index: 0.7699530516431925
[I 2024-04-16 00:49:33,018] Trial 0 finished with value: 0.7575203363732566 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7575203363732566.
Fold 1 C-index: 0.7445887445887446
Fold 2 C-index: 0.8169642857142857
Fold 3 C-index: 0.75
Fold 4 C-index: 0.6708860759493671
Fold 5 C-index: 0.7699530516431925
[I 2024-04-16 00:49:37,392] Trial 1 finished with value: 0.750478431579118 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'max_features': 'sqrt', 'mi

Fold 1 C-index: 0.7597402597402597
Fold 2 C-index: 0.8526785714285714
Fold 3 C-index: 0.7867647058823529
Fold 4 C-index: 0.7236286919831224
Fold 5 C-index: 0.8028169014084507
[I 2024-04-16 00:50:27,203] Trial 17 finished with value: 0.7851258260885514 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 15, 'min_samples_leaf': 15, 'max_depth': 17, 'n_estimators': 106, 'oob_score': True, 'max_samples': 0.9817072304699028, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.16544550098450153, 'warm_start': True}. Best is trial 14 with value: 0.7907144637948861.
Fold 1 C-index: 0.7575757575757576
Fold 2 C-index: 0.8504464285714286
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.7194092827004219
Fold 5 C-index: 0.7981220657276995
[I 2024-04-16 00:50:28,286] Trial 18 finished with value: 0.7834440402483949 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 19, 'max_depth': 7, 'n_estimators': 118, 'oob_score': True, 'max_samples': 0.986237414920

Fold 1 C-index: 0.7575757575757576
Fold 2 C-index: 0.8348214285714286
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.7742616033755274
Fold 5 C-index: 0.8309859154929577
[I 2024-04-16 00:50:36,394] Trial 32 finished with value: 0.8032544311992128 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 14, 'min_samples_leaf': 7, 'max_depth': 9, 'n_estimators': 78, 'oob_score': True, 'max_samples': 0.7397628776532771, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.04641084093066284, 'warm_start': True}. Best is trial 32 with value: 0.8032544311992128.
Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.8348214285714286
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.770042194092827
Fold 5 C-index: 0.8309859154929577
[I 2024-04-16 00:50:37,066] Trial 33 finished with value: 0.7999277380363321 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 14, 'min_samples_leaf': 7, 'max_depth': 12, 'n_estimators': 77, 'oob_score': True, 'max_samples': 0.7321956680996766, '

...

Fold 5 IBS: 0.17698987901902175
[I 2024-04-16 00:52:41,119] Trial 31 finished with value: 0.1756223060197624 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 17, 'min_samples_leaf': 13, 'max_depth': 1, 'n_estimators': 495, 'oob_score': False, 'max_samples': 0.6141886010044044, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.050889504115644586}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.17921499719523448
Fold 2 IBS: 0.14457659998302025
Fold 3 IBS: 0.18892068480933402
Fold 4 IBS: 0.1881785321534129
Fold 5 IBS: 0.17366220512930025
[I 2024-04-16 00:52:43,378] Trial 32 finished with value: 0.17491060385406038 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 19, 'min_samples_leaf': 11, 'max_depth': 2, 'n_estimators': 473, 'oob_score': False, 'max_samples': 0.7145937356077748, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.040680386261001684}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.17839650482493216
Fold 2 IBS: 0.

Fold 1 IBS: 0.16980857091041326
Fold 2 IBS: 0.15719115142034723
Fold 3 IBS: 0.1767449129839921
Fold 4 IBS: 0.19510331227382113
Fold 5 IBS: 0.1831755952242518
[I 2024-04-16 00:53:27,703] Trial 47 finished with value: 0.17640470856256513 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 20, 'min_samples_leaf': 11, 'max_depth': 2, 'n_estimators': 479, 'oob_score': False, 'max_samples': 0.8914383434724882, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.24824101632322468}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.19349637716238946
Fold 2 IBS: 0.18021514079410297
Fold 3 IBS: 0.1797766992061227
Fold 4 IBS: 0.20805913242700583
Fold 5 IBS: 0.19958771263889286
[I 2024-04-16 00:53:30,473] Trial 48 finished with value: 0.19222701244570276 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 14, 'min_samples_leaf': 6, 'max_depth': 3, 'n_estimators': 439, 'oob_score': False, 'max_samples': 0.9967788477130068, 'max_features': 'auto', 'min_weight_fraction_

Fold 5 IBS: 0.17601995444169838
[I 2024-04-16 00:54:14,738] Trial 62 finished with value: 0.1766496500814655 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 16, 'min_samples_leaf': 9, 'max_depth': 3, 'n_estimators': 298, 'oob_score': True, 'max_samples': 0.6969767031486694, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.11947149985769817}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.17325965795999854
Fold 2 IBS: 0.16035062956465257
Fold 3 IBS: 0.17539882197990753
Fold 4 IBS: 0.193362443315604
Fold 5 IBS: 0.1796554283287958
[I 2024-04-16 00:54:17,235] Trial 63 finished with value: 0.17640539622979168 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 15, 'min_samples_leaf': 10, 'max_depth': 1, 'n_estimators': 311, 'oob_score': True, 'max_samples': 0.6304198495967799, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.16702154024396243}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.17417602273955624
Fold 2 IBS: 0.1480464

Fold 1 IBS: 0.16818314645426713
Fold 2 IBS: 0.15369061634554518
Fold 3 IBS: 0.18213459050433092
Fold 4 IBS: 0.19123211966917278
Fold 5 IBS: 0.17979973767912885
[I 2024-04-16 00:55:43,556] Trial 78 finished with value: 0.17500804213048896 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 17, 'min_samples_leaf': 11, 'max_depth': 2, 'n_estimators': 487, 'oob_score': False, 'max_samples': 0.4897248144615137, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.053996872945124036}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.1682237549262069
Fold 2 IBS: 0.15295334400832536
Fold 3 IBS: 0.18214608764478374
Fold 4 IBS: 0.19238806563941324
Fold 5 IBS: 0.18028732465861996
[I 2024-04-16 00:55:48,965] Trial 79 finished with value: 0.17519971537546983 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 17, 'min_samples_leaf': 11, 'max_depth': 11, 'n_estimators': 500, 'oob_score': False, 'max_samples': 0.49575398669591153, 'max_features': 'auto', 'min_weight_frac

Fold 5 IBS: 0.1747309238915691
[I 2024-04-16 00:56:58,955] Trial 93 finished with value: 0.1752048898255005 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 19, 'min_samples_leaf': 10, 'max_depth': 4, 'n_estimators': 437, 'oob_score': False, 'max_samples': 0.5630698614554802, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.0944985972942633}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.17026943747099219
Fold 2 IBS: 0.15255352423829832
Fold 3 IBS: 0.18160069813968083
Fold 4 IBS: 0.19096533404342483
Fold 5 IBS: 0.1822030877543707
[I 2024-04-16 00:57:02,017] Trial 94 finished with value: 0.17551841632935336 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 18, 'min_samples_leaf': 11, 'max_depth': 2, 'n_estimators': 410, 'oob_score': False, 'max_samples': 0.5751544947591161, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.14409687866880386}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.17475332961241374
Fold 2 IBS: 0.1452

In [48]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [49]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.846
train_ibs:  0.171


#### Test

In [50]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [51]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=13, max_features=None, max_leaf_nodes=12,
                     max_samples=0.48042970450395145, min_samples_leaf=1,
                     min_samples_split=4,
                     min_weight_fraction_leaf=0.0025914663927405594,
                     n_estimators=289, random_state=123, warm_start=True)

test_cindex:  0.602


RandomSurvivalForest(max_depth=1, max_features='auto', max_leaf_nodes=18,
                     max_samples=0.7351810575897255, min_samples_leaf=11,
                     min_samples_split=2,
                     min_weight_fraction_leaf=0.008231293935378081,
                     n_estimators=2, random_state=123)

test_ibs:  0.233


In [52]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [53]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 00:57:18,108] A new study created in memory with name: no-name-2faa02ae-db1e-4fa3-84ec-ef91420caf17


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.70995670995671
Fold 2 C-index: 0.8348214285714286
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.7257383966244726
Fold 5 C-index: 0.755868544600939
[I 2024-04-16 00:57:19,062] Trial 0 finished with value: 0.763120153205612 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.763120153205612.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 00:57:20,901] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531}. Be

Fold 1 C-index: 0.6645021645021645
Fold 2 C-index: 0.8191964285714286
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.7088607594936709
Fold 5 C-index: 0.676056338028169
[I 2024-04-16 00:57:40,788] Trial 16 finished with value: 0.7335270596877141 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8750346354626457, 'min_weight_fraction_leaf': 0.4093818399278544}. Best is trial 12 with value: 0.7656344262692516.
Fold 1 C-index: 0.7445887445887446
Fold 2 C-index: 0.8348214285714286
Fold 3 C-index: 0.7794117647058824
Fold 4 C-index: 0.7046413502109705
Fold 5 C-index: 0.7323943661971831
[I 2024-04-16 00:57:41,397] Trial 17 finished with value: 0.7591715308548418 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 10, 'min_samples_leaf': 8, 'max_depth': 12, 'n_estimators': 318, 'oob_score': False, 'warm_start': True, 'max_featu

Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.84375
Fold 3 C-index: 0.7843137254901961
Fold 4 C-index: 0.7046413502109705
Fold 5 C-index: 0.7276995305164319
[I 2024-04-16 00:57:57,138] Trial 31 finished with value: 0.7601328692954678 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 16, 'min_samples_leaf': 15, 'max_depth': 3, 'n_estimators': 375, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.9462932212483206, 'min_weight_fraction_leaf': 0.12021009726526694}. Best is trial 19 with value: 0.7663281634231028.
Fold 1 C-index: 0.7272727272727273
Fold 2 C-index: 0.84375
Fold 3 C-index: 0.7794117647058824
Fold 4 C-index: 0.7215189873417721
Fold 5 C-index: 0.7323943661971831
[I 2024-04-16 00:57:57,997] Trial 32 finished with value: 0.760869569103513 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 17, 'min_samples_leaf': 14, 'max_depth': 3, 'n_estimators': 462, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_sa

...

Fold 1 IBS: 0.18923073618097216
Fold 2 IBS: 0.17632006586851645
Fold 3 IBS: 0.18707067058836963
Fold 4 IBS: 0.19307240934736852
Fold 5 IBS: 0.19343354994654252
[I 2024-04-16 01:00:36,398] Trial 30 finished with value: 0.18782548638635385 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 3, 'min_samples_leaf': 4, 'max_depth': 12, 'n_estimators': 355, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7262614295600427, 'min_weight_fraction_leaf': 0.020774312173092817}. Best is trial 23 with value: 0.18218668005091396.
Fold 1 IBS: 0.1949036294792145
Fold 2 IBS: 0.1886371865011974
Fold 3 IBS: 0.19117863815442443
Fold 4 IBS: 0.2001198684681372
Fold 5 IBS: 0.19929856633981852
[I 2024-04-16 01:00:40,793] Trial 31 finished with value: 0.19482757778855841 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 2, 'min_samples_leaf': 4, 'max_depth': 12, 'n_estimators': 396, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.

Fold 1 IBS: 0.18107213977138975
Fold 2 IBS: 0.17375539456626857
Fold 3 IBS: 0.18442566650093445
Fold 4 IBS: 0.19068424307356363
Fold 5 IBS: 0.18831741970696433
[I 2024-04-16 01:02:10,003] Trial 45 finished with value: 0.18365097272382416 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 11, 'min_samples_leaf': 2, 'max_depth': 17, 'n_estimators': 366, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.9992649320581628, 'min_weight_fraction_leaf': 0.15624439720920977}. Best is trial 44 with value: 0.17800229474539986.
Fold 1 IBS: 0.19953982633918327
Fold 2 IBS: 0.19967504623742535
Fold 3 IBS: 0.19245042867286613
Fold 4 IBS: 0.2085347403492104
Fold 5 IBS: 0.20542311562702958
[I 2024-04-16 01:02:17,388] Trial 46 finished with value: 0.20112463144514292 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 10, 'min_samples_leaf': 4, 'max_depth': 15, 'n_estimators': 471, 'oob_score': True, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.8

Fold 1 IBS: 0.17659457791938485
Fold 2 IBS: 0.15948702371575116
Fold 3 IBS: 0.18725320428102982
Fold 4 IBS: 0.18157027728642194
Fold 5 IBS: 0.18138733472020935
[I 2024-04-16 01:03:53,732] Trial 60 finished with value: 0.17725848358455942 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 5, 'min_samples_leaf': 6, 'max_depth': 15, 'n_estimators': 346, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.7897702249078459, 'min_weight_fraction_leaf': 0.017004106952235656}. Best is trial 57 with value: 0.1764459724345519.
Fold 1 IBS: 0.17815851130122365
Fold 2 IBS: 0.15937759601308119
Fold 3 IBS: 0.18671233531081954
Fold 4 IBS: 0.18030514263303932
Fold 5 IBS: 0.18041925981213724
[I 2024-04-16 01:04:02,627] Trial 61 finished with value: 0.1769945690140602 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 5, 'min_samples_leaf': 6, 'max_depth': 15, 'n_estimators': 340, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.861

Fold 1 IBS: 0.21400073415496554
Fold 2 IBS: 0.2208296768873117
Fold 3 IBS: 0.20504439468604338
Fold 4 IBS: 0.22482948519570425
Fold 5 IBS: 0.21793226667188703
[I 2024-04-16 01:06:30,835] Trial 75 finished with value: 0.21652731151918242 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 6, 'min_samples_leaf': 7, 'max_depth': 11, 'n_estimators': 301, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.28939609623192764, 'min_weight_fraction_leaf': 0.38792353869222}. Best is trial 73 with value: 0.17587600438953827.
Fold 1 IBS: 0.18092605908791307
Fold 2 IBS: 0.1626714178491272
Fold 3 IBS: 0.1886398695722679
Fold 4 IBS: 0.18093264680250482
Fold 5 IBS: 0.18045471816506836
[I 2024-04-16 01:06:40,941] Trial 76 finished with value: 0.17872494229537628 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 4, 'min_samples_leaf': 5, 'max_depth': 14, 'n_estimators': 229, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.82756

Fold 1 IBS: 0.18417873567489768
Fold 2 IBS: 0.17215507074054423
Fold 3 IBS: 0.18759974758071274
Fold 4 IBS: 0.1889870504957803
Fold 5 IBS: 0.18950903071510605
[I 2024-04-16 01:09:52,546] Trial 90 finished with value: 0.1844859270414082 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 4, 'min_samples_leaf': 4, 'max_depth': 10, 'n_estimators': 262, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.9722689668575789, 'min_weight_fraction_leaf': 0.013698489831834436}. Best is trial 73 with value: 0.17587600438953827.
Fold 1 IBS: 0.17828049657703648
Fold 2 IBS: 0.15670442146325114
Fold 3 IBS: 0.1885595963781235
Fold 4 IBS: 0.17837564239035944
Fold 5 IBS: 0.17924664597888262
[I 2024-04-16 01:10:07,091] Trial 91 finished with value: 0.17623336055753064 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 5, 'max_depth': 15, 'n_estimators': 308, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.83

In [54]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [55]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.791
train_ibs:  0.176


#### Test

In [56]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [57]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=19, max_features=1, max_leaf_nodes=12,
                   max_samples=0.21404335777324493, min_samples_leaf=1,
                   min_samples_split=4,
                   min_weight_fraction_leaf=0.0013133682808468272,
                   n_estimators=405, random_state=123, warm_start=True)

C-index score: 0.572


ExtraSurvivalTrees(max_depth=14, max_features=None, max_leaf_nodes=6,
                   max_samples=0.8396800309305413, min_samples_leaf=5,
                   min_samples_split=5,
                   min_weight_fraction_leaf=0.02494499808293557,
                   n_estimators=305, oob_score=True, random_state=123)

IBS: 0.221


In [58]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [59]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 01:11:43,532] A new study created in memory with name: no-name-e1a887f7-ad0a-4b6d-9ad9-990f03ba831f


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 01:12:32,244] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 01:12:55,311] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 01:38:10,622] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7610259439316011.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 01:41:34,742] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:03:40,524] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.8840318412875596, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.27382248438555523, 'n_estimators': 385, 'criterion': 'squared_error', 'ccp_alpha': 2.0183033060186855, 'min_weight_fraction_leaf': 0.33643534713187806, 'max_features': 'auto', 'min_impurity_decrease': 5.889654690360788e-06, 'validation_fraction': 0.8062622793646869, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 9 with value: 0.7610259439316011.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:04:57,750] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.7565765917190008, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.4300954216773497, 'n_estimators': 440, 'criterion': 'friedman

...

Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609557
[I 2024-04-16 04:00:57,358] Trial 44 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9992870370700113, 'learning_rate': 0.022847552015173876, 'dropout_rate': 0.1556807870961761, 'n_estimators': 451, 'criterion': 'squared_error', 'ccp_alpha': 1.6646055539220843, 'min_weight_fraction_leaf': 0.1403134453903068, 'max_features': 'auto', 'min_impurity_decrease': 2.7552659293421345e-07, 'validation_fraction': 0.8820166309308186, 'min_samples_split': 17, 'max_leaf_nodes': 14, 'min_samples_leaf': 10, 'max_depth': 2}. Best is trial 42 with value: 0.21286195228463267.
Fold 1 IBS: 0.21224782790603694
Fold 2 IBS: 0.2189260608342495
Fold 3 IBS: 0.20307124042767
Fold 4 IBS: 0.22313607826909793
Fold 5 IBS: 0.216439921823279
[I 2024-04-16 04:01:30,318] Trial 45 finished with value: 0.2147642258520667 and parameters: {'subsample': 0.8888212863898438, 'learning_rate': 0.0152498321107806

Fold 3 IBS: 0.20286393718880646
Fold 4 IBS: 0.22391249179729633
Fold 5 IBS: 0.21519882725417372
[I 2024-04-16 04:09:19,445] Trial 55 finished with value: 0.21426660724128474 and parameters: {'subsample': 0.9703353679292269, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.2676210325613897, 'n_estimators': 481, 'criterion': 'squared_error', 'ccp_alpha': 0.036860238643527846, 'min_weight_fraction_leaf': 0.21798842867076448, 'max_features': None, 'min_impurity_decrease': 4.086647023052284e-07, 'validation_fraction': 0.8868940629916056, 'min_samples_split': 18, 'max_leaf_nodes': 15, 'min_samples_leaf': 10, 'max_depth': 4}. Best is trial 42 with value: 0.21286195228463267.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609582
[I 2024-04-16 04:10:17,902] Trial 56 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.9556274374724505, 'learning_rate': 0.022777236

Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609563
[I 2024-04-16 04:13:22,875] Trial 67 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.922035492452401, 'learning_rate': 0.02138812894899817, 'dropout_rate': 0.2208120967013012, 'n_estimators': 121, 'criterion': 'squared_error', 'ccp_alpha': 0.29393333981111347, 'min_weight_fraction_leaf': 0.09999708192212713, 'max_features': None, 'min_impurity_decrease': 1.467292826198341e-07, 'validation_fraction': 0.8460725291615868, 'min_samples_split': 20, 'max_leaf_nodes': 14, 'min_samples_leaf': 10, 'max_depth': 12}. Best is trial 57 with value: 0.21250989041794505.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-16 04:13:24,918] Trial 68 finished with value: 0.21659054862241586 and parameters:

Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.2181243152560957
[I 2024-04-16 04:15:06,119] Trial 78 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9202091045129451, 'learning_rate': 0.019797687069268054, 'dropout_rate': 0.12670174450467558, 'n_estimators': 79, 'criterion': 'friedman_mse', 'ccp_alpha': 0.2910342652485194, 'min_weight_fraction_leaf': 0.15713001580184646, 'max_features': 'log2', 'min_impurity_decrease': 6.996091386072907e-06, 'validation_fraction': 0.7499357618644306, 'min_samples_split': 15, 'max_leaf_nodes': 15, 'min_samples_leaf': 15, 'max_depth': 16}. Best is trial 70 with value: 0.21173544073366624.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609582
[I 2024-04-16 04:15:08,047] Trial 79 finished with value: 0.21659054862241592 and parameters

Fold 2 IBS: 0.20607296701907554
Fold 3 IBS: 0.19811121797640907
Fold 4 IBS: 0.22011942798029224
Fold 5 IBS: 0.20776003972546755
[I 2024-04-16 04:16:03,688] Trial 89 finished with value: 0.2072255779812277 and parameters: {'subsample': 0.9081334064688743, 'learning_rate': 0.032246585462716706, 'dropout_rate': 0.10084377376256726, 'n_estimators': 92, 'criterion': 'friedman_mse', 'ccp_alpha': 0.019065157478907357, 'min_weight_fraction_leaf': 0.21843803937299006, 'max_features': None, 'min_impurity_decrease': 6.552978044824526e-05, 'validation_fraction': 0.8292287698601738, 'min_samples_split': 18, 'max_leaf_nodes': 15, 'min_samples_leaf': 9, 'max_depth': 3}. Best is trial 89 with value: 0.2072255779812277.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609566
[I 2024-04-16 04:16:08,952] Trial 90 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.8590448841763807, 

In [60]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [61]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.765
train_ibs:  0.207


#### Test

In [62]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [63]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.046986154574499193,
                                 dropout_rate=0.9141084507237711,
                                 learning_rate=0.005228819896792163,
                                 max_leaf_nodes=13,
                                 min_impurity_decrease=0.00017501601948817654,
                                 min_samples_leaf=15, min_samples_split=11,
                                 min_weight_fraction_leaf=0.2724947676380759,
                                 n_estimators=395, random_state=123,
                                 subsample=0.6128161811459047,
                                 validation_fraction=0.542037097833947)

C-index score: 0.579


GradientBoostingSurvivalAnalysis(ccp_alpha=0.019065157478907357,
                                 dropout_rate=0.10084377376256726,
                                 learning_rate=0.032246585462716706,
                                 max_leaf_nodes=15,
                                 min_impurity_decrease=6.552978044824526e-05,
                                 min_samples_leaf=9, min_samples_split=18,
                                 min_weight_fraction_leaf=0.21843803937299006,
                                 n_estimators=92, random_state=123,
                                 subsample=0.9081334064688743,
                                 validation_fraction=0.8292287698601738)

IBS: 0.218


In [64]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [65]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 04:16:56,699] A new study created in memory with name: no-name-7b173a3e-5d24-4839-854e-8084fb218ec5


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7647058823529411
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7323943661971831
[I 2024-04-16 04:16:58,056] Trial 0 finished with value: 0.6944636137409307 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6944636137409307.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7598039215686274
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7323943661971831
[I 2024-04-16 04:17:07,843] Trial 1 finished with value: 0.6934832215840679 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6944636137409307.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7647058823529411
Fold 4 C-index: 0.6708860759493671
Fold 5 C-ind

Fold 1 C-index: 0.6060606060606061
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.7794117647058824
Fold 4 C-index: 0.6835443037974683
Fold 5 C-index: 0.7417840375586855
[I 2024-04-16 04:18:27,552] Trial 19 finished with value: 0.7112672852816713 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.7196046509277619, 'n_estimators': 431, 'learning_rate': 0.08300323114610605}. Best is trial 11 with value: 0.7117135919830906.
Fold 1 C-index: 0.5584415584415584
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7696078431372549
Fold 4 C-index: 0.6751054852320675
Fold 5 C-index: 0.7417840375586855
[I 2024-04-16 04:18:34,244] Trial 20 finished with value: 0.6989877848739133 and parameters: {'subsample': 0.2630057481431337, 'dropout_rate': 0.7628923189846966, 'n_estimators': 435, 'learning_rate': 0.08036525450908355}. Best is trial 11 with value: 0.7117135919830906.
Fold 1 C-index: 0.6060606060606061
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.7794117647058824
Fold 4 C-index

Fold 1 C-index: 0.5757575757575758
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.6919831223628692
Fold 5 C-index: 0.7370892018779343
[I 2024-04-16 04:19:20,177] Trial 38 finished with value: 0.7089841872825892 and parameters: {'subsample': 0.1499958194858822, 'dropout_rate': 0.7725266545363573, 'n_estimators': 192, 'learning_rate': 0.09420011192077271}. Best is trial 31 with value: 0.712247677438534.
Fold 1 C-index: 0.5584415584415584
Fold 2 C-index: 0.7544642857142857
Fold 3 C-index: 0.7696078431372549
Fold 4 C-index: 0.6708860759493671
Fold 5 C-index: 0.7464788732394366
[I 2024-04-16 04:19:24,800] Trial 39 finished with value: 0.6999757272963805 and parameters: {'subsample': 0.3156029735189495, 'dropout_rate': 0.6953650320328696, 'n_estimators': 339, 'learning_rate': 0.06456811545507721}. Best is trial 31 with value: 0.712247677438534.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7647058823529411
Fold 4 C-index: 0

...

Fold 2 IBS: 0.17580127670955806
Fold 3 IBS: 0.17528921045094445
Fold 4 IBS: 0.19335846310954638
Fold 5 IBS: 0.18393908760893649
[I 2024-04-16 04:26:45,914] Trial 19 finished with value: 0.18631180066870817 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.9930703343219778, 'n_estimators': 51, 'learning_rate': 0.04096883532946153}. Best is trial 19 with value: 0.18631180066870817.
Fold 1 IBS: 0.20469079919009717
Fold 2 IBS: 0.18025608024789408
Fold 3 IBS: 0.17685175698416405
Fold 4 IBS: 0.19581523101059145
Fold 5 IBS: 0.18727333669387985
[I 2024-04-16 04:26:46,308] Trial 20 finished with value: 0.1889774408253253 and parameters: {'subsample': 0.1435040576337751, 'dropout_rate': 0.7933084651006226, 'n_estimators': 43, 'learning_rate': 0.04127909023805986}. Best is trial 19 with value: 0.18631180066870817.
Fold 1 IBS: 0.20189804584228907
Fold 2 IBS: 0.18605742016473745
Fold 3 IBS: 0.18348619158171814
Fold 4 IBS: 0.20331810366947325
Fold 5 IBS: 0.1908481491711361
[I 2024

Fold 3 IBS: 0.17929942687774186
Fold 4 IBS: 0.19699676923820636
Fold 5 IBS: 0.17398211460958732
[I 2024-04-16 04:27:06,146] Trial 38 finished with value: 0.18583868121614 and parameters: {'subsample': 0.10045702238474481, 'dropout_rate': 0.13272164755980653, 'n_estimators': 82, 'learning_rate': 0.0684395247111694}. Best is trial 38 with value: 0.18583868121614.
Fold 1 IBS: 0.22888158579155815
Fold 2 IBS: 0.1778204923038366
Fold 3 IBS: 0.18090871975165124
Fold 4 IBS: 0.20677627347434813
Fold 5 IBS: 0.17142461059239106
[I 2024-04-16 04:27:06,989] Trial 39 finished with value: 0.19316233638275704 and parameters: {'subsample': 0.1669627464432204, 'dropout_rate': 0.11471626893495929, 'n_estimators': 82, 'learning_rate': 0.06892741183938003}. Best is trial 38 with value: 0.18583868121614.
Fold 1 IBS: 0.26127361158239537
Fold 2 IBS: 0.2702117097774595
Fold 3 IBS: 0.21315070084074397
Fold 4 IBS: 0.2757159479700989
Fold 5 IBS: 0.21996299901762018
[I 2024-04-16 04:27:08,015] Trial 40 finished wi

Fold 4 IBS: 0.21234116935596248
Fold 5 IBS: 0.17496721369462195
[I 2024-04-16 04:27:27,316] Trial 57 finished with value: 0.1916521637064573 and parameters: {'subsample': 0.22011128988939735, 'dropout_rate': 0.17296438704055608, 'n_estimators': 49, 'learning_rate': 0.07586937026396535}. Best is trial 41 with value: 0.1834742491042333.
Fold 1 IBS: 0.20712451000200607
Fold 2 IBS: 0.1872772816103758
Fold 3 IBS: 0.17693753937978524
Fold 4 IBS: 0.20322441833559401
Fold 5 IBS: 0.1877441673818591
[I 2024-04-16 04:27:27,651] Trial 58 finished with value: 0.19246158334192404 and parameters: {'subsample': 0.2849097248903497, 'dropout_rate': 0.36573681723837637, 'n_estimators': 31, 'learning_rate': 0.060250528734790795}. Best is trial 41 with value: 0.1834742491042333.
Fold 1 IBS: 0.20840523189667032
Fold 2 IBS: 0.16954415076351176
Fold 3 IBS: 0.17248829895886533
Fold 4 IBS: 0.19198239637835499
Fold 5 IBS: 0.17570932494225114
[I 2024-04-16 04:27:28,655] Trial 59 finished with value: 0.18362588058

Fold 3 IBS: 0.23054394267833275
Fold 4 IBS: 0.27895191507279604
Fold 5 IBS: 0.25683011697051605
[I 2024-04-16 04:27:36,769] Trial 76 finished with value: 0.2672213949040877 and parameters: {'subsample': 0.7815899098735377, 'dropout_rate': 0.15568721954888723, 'n_estimators': 333, 'learning_rate': 0.038380847630475584}. Best is trial 41 with value: 0.1834742491042333.
Fold 1 IBS: 0.2134266504233145
Fold 2 IBS: 0.1806008546727606
Fold 3 IBS: 0.1727394209991852
Fold 4 IBS: 0.2067021380764206
Fold 5 IBS: 0.1811886564373895
[I 2024-04-16 04:27:51,754] Trial 77 finished with value: 0.19093154412181407 and parameters: {'subsample': 0.22380896794758492, 'dropout_rate': 0.223235951943684, 'n_estimators': 93, 'learning_rate': 0.030144736081125657}. Best is trial 41 with value: 0.1834742491042333.
Fold 1 IBS: 0.20107212380951917
Fold 2 IBS: 0.18248092538711716
Fold 3 IBS: 0.17884990049571556
Fold 4 IBS: 0.19734752701464475
Fold 5 IBS: 0.18708437778913986
[I 2024-04-16 04:27:52,372] Trial 78 finis

Fold 3 IBS: 0.1730444856942059
Fold 4 IBS: 0.2078118105594406
Fold 5 IBS: 0.1726855062527798
[I 2024-04-16 04:28:24,158] Trial 95 finished with value: 0.1887822960208532 and parameters: {'subsample': 0.20603499818629928, 'dropout_rate': 0.1146228273465264, 'n_estimators': 91, 'learning_rate': 0.042392461808359164}. Best is trial 93 with value: 0.18152261659638766.
Fold 1 IBS: 0.20648347961858549
Fold 2 IBS: 0.20643794993984038
Fold 3 IBS: 0.19466029615625455
Fold 4 IBS: 0.21236891856726114
Fold 5 IBS: 0.20653743680416023
[I 2024-04-16 04:28:24,426] Trial 96 finished with value: 0.20529761621722034 and parameters: {'subsample': 0.11517878134212854, 'dropout_rate': 0.4933732563935224, 'n_estimators': 15, 'learning_rate': 0.035798638536663166}. Best is trial 93 with value: 0.18152261659638766.
Fold 1 IBS: 0.22872724174787454
Fold 2 IBS: 0.18940280361943704
Fold 3 IBS: 0.17802392222300426
Fold 4 IBS: 0.21468859489239034
Fold 5 IBS: 0.17630931950961182
[I 2024-04-16 04:28:25,272] Trial 97 f

In [66]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [67]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.714
train_ibs:  0.182


#### Test

In [68]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [69]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.6202793429199533,
                                              learning_rate=0.08136047667579,
                                              n_estimators=494,
                                              random_state=123,
                                              subsample=0.11713675251393994)

C-index score: 0.573


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.11978813044506781,
                                              learning_rate=0.039463829092430826,
                                              n_estimators=96, random_state=123,
                                              subsample=0.12098460820177237)

IBS: 0.231


In [70]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [71]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.846,1.0
ExtraSurvivalTrees,0.791,2.0
GradientBoosting,0.765,3.0
CoxElastic,0.737,4.0
CoxPH,0.736,5.5
CoxLasso,0.736,5.5
ComponentwiseGradientBoosting,0.714,7.0
CoxRidge,0.695,8.0


In [72]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
CoxLasso,0.171,2.0
CoxElastic,0.171,2.0
Randomsurvivalforest,0.171,2.0
CoxPH,0.172,4.0
ExtraSurvivalTrees,0.176,5.0
ComponentwiseGradientBoosting,0.182,6.0
GradientBoosting,0.207,7.0
CoxRidge,0.217,8.0


In [73]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
Randomsurvivalforest,0.602,1.0
GradientBoosting,0.579,2.0
CoxRidge,0.577,3.0
ComponentwiseGradientBoosting,0.573,4.0
ExtraSurvivalTrees,0.572,5.0
CoxPH,0.567,6.0
CoxLasso,0.566,7.5
CoxElastic,0.566,7.5


In [74]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
GradientBoosting,0.218,1.0
CoxRidge,0.221,2.5
ExtraSurvivalTrees,0.221,2.5
ComponentwiseGradientBoosting,0.231,4.0
Randomsurvivalforest,0.233,5.0
CoxLasso,0.257,6.5
CoxElastic,0.257,6.5
CoxPH,0.259,8.0


In [75]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = 'path_to_your_folder/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d2/os/robust/rent/' 

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d2_os_robust_rent_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [76]:
from datetime import date

current_date = date.today()
print(current_date)

2024-04-16
